<a href="https://colab.research.google.com/github/jhajagos/SupportingConceptSetGeneration/blob/main/UMLS_API_with_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Environment

In [2]:
import requests
import pprint
import json

In [3]:
!pip install -q -U google-generativeai

In [4]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [5]:
UMLS_API_KEY = userdata.get('UMLS_API_KEY')
UMLS_BASE_URL= "https://uts-ws.nlm.nih.gov/rest"

## UMLS API

In [6]:
t = """/search/{version}|Retrieves CUIs when searching by term or code
  /content/{version}/CUI/{CUI}|Retrieves information about a known CUI
  /content/{version}/CUI/{CUI}/atoms|Retrieves atoms and information about atoms for a known CUI
  /content/{version}/CUI/{CUI}/definitions|Retrieves definitions for a known CUI
  /content/{version}/CUI/{CUI}/relations|Retrieves NLM-asserted relationships for a known CUI
  /content/{version}/source/{source}/{id}|Retrieves information about a known source-asserted identifier
  /content/{version}/source/{source}/{id}/atoms|Retrieves information about atoms for a known source-asserted identifier
  /content/{version}/source/{source}/{id}/parents|Retrieves immediate parents of a source-asserted identifier
  /content/{version}/source/{source}/{id}/children|Retrieves immediate children of a source-asserted identifier
  /content/{version}/source/{source}/{id}/ancestors|Retrieves all ancestors of a source-asserted identifier
  /content/{version}/source/{source}/{id}/descendants|Retrieves all descendants of a source-asserted identifier
  /content/{version}/source/{source}/{id}/relations|Retrieves all relationships of a source-asserted identifier
  /content/{version}/source/{source}/{id}/attributes|Retrieves information about source-asserted attributes
  /semantic-network/{version}/TUI/{id}|Retrieves information for a known Semantic Type identifier (TUI)
  /crosswalk/{version}/source/{source}/{id}|Retrieves all source-asserted identifiers that share a UMLS CUI with a particular code"""
api_templates = [l.strip().split("|") for l in t.split("\n")]

In [7]:
import pandas as pd
pd.DataFrame(api_templates, columns=["Path", "Description"])

,Path,Description
0,/search/{version},Retrieves CUIs when searching by term or code
1,/content/{version}/CUI/{CUI},Retrieves information about a known CUI
2,/content/{version}/CUI/{CUI}/atoms,Retrieves atoms and information about atoms fo...
3,/content/{version}/CUI/{CUI}/definitions,Retrieves definitions for a known CUI
4,/content/{version}/CUI/{CUI}/relations,Retrieves NLM-asserted relationships for a kno...
5,/content/{version}/source/{source}/{id},Retrieves information about a known source-ass...
6,/content/{version}/source/{source}/{id}/atoms,Retrieves information about atoms for a known ...
7,/content/{version}/source/{source}/{id}/parents,Retrieves immediate parents of a source-assert...
8,/content/{version}/source/{source}/{id}/children,Retrieves immediate children of a source-asser...
9,/content/{version}/source/{source}/{id}/ancestors,Retrieves all ancestors of a source-asserted i...


In [8]:
import time
import logging

logging.basicConfig(level=logging.INFO)

def process_relationships(relationships):
  """Build a list of relationships from the API response"""
  relationship_list = []
  for relation in relationships:
    relationship_list += [{"relationship_source": relation['rootSource'],
                            "relationship_type": relation['relationLabel'],
                            "relationship_type_description": relation['additionalRelationLabel'],
                            "relationship_target": relation['relatedIdName'],
                            "relationship_target_code": relation["relatedId"].split("/")[-1]
                          }]
  return relationship_list

def process_hierarchial_term(terms):
  result_list = []
  for term in terms:
    result_list += [{
        "id": term["ui"],
        "name": term["name"]}
    ]
  return result_list


def umls_request(url, api_key=UMLS_API_KEY, additional_params={}):
  """Make a request to the UMLS API and return the response"""
  start_time = time.time()

  params = {"apiKey": api_key}
  params.update(additional_params)

  logging.info(f"Making request to {url} with params {params}")
  response = requests.get(url, params=params).json()

  end_time = time.time()
  logging.info(f"Request took {end_time - start_time} seconds")

  return response


def get_vocabularies():
  rl = umls_request(UMLS_BASE_URL + "/metadata/current/sources")
  return {r["abbreviation"]: r["expandedForm"] for r in rl['result']}


def get_vocabulary_languages():
  rl = umls_request(UMLS_BASE_URL + "/metadata/current/sources")
  return {r["abbreviation"]: r["language"]["expandedForm"] for r in rl['result']}


def get_code_source_information(code, source="ICD10CM", version = "current"):
  """For given terms in a source vocabulary gets context around the term and
    returns results as a dictionary."""


  languages = get_vocabulary_languages() # Get language

  r_obj = umls_request(UMLS_BASE_URL + f"/content/{version}/source/{source}/{code}")

  if "result" not in r_obj:
    return None
  else:
    r = r_obj["result"]

    name = r["name"]

    attributes_dict = {} # Gets a term attributes (MRSAT table)
    if r["attributes"] == "NONE":
      pass
    else:
      attributes = umls_request(r["attributes"], additional_params={"pageSize": 100}) # TODO: Add paging
      if "result" in attributes:
        for attribute in attributes["result"]:
          attributes_dict[attribute["name"]] = attribute["value"]

    relationship_list = [] # Gets term relationships (MRREL)
    if r["relations"] == "NONE":
      pass
    else:
      relationships = umls_request(r["relations"], additional_params={"pageSize": 100})
      if "pageCount" in relationships:
        relationship_list = process_relationships(relationships["result"])
        if relationships["pageCount"] > 1:
          for i in range(2,relationships["pageCount"]+1):
            i_relationships = umls_request(r["relations"], additional_params={"pageSize": 100, "pageNumber": i})
            relationship_list += process_relationships(i_relationships["result"])

    # Get CUIs associated with the term (MRCONSO)
    concept_url = r["concepts"]
    concepts_obj = umls_request(concept_url)

    # Get parent and children terms

    parent_list = []
    children_list = []

    parents = r["parents"]
    children = r["children"]

    if parents != "NONE":
      parent_obj = umls_request(parents)
      parent_list = process_hierarchial_term(parent_obj["result"])

    if children != "NONE":
      children_obj = umls_request(children)
      children_list = process_hierarchial_term(children_obj["result"])

    ancestors_list = []
    descendants_list = []

    ancestors = r["ancestors"]
    descendants = r["descendants"]

    if ancestors != "NONE":
      ancestors_obj = umls_request(ancestors)
      ancestors_list = process_hierarchial_term(ancestors_obj["result"])

    if descendants != "NONE":
      descendants_obj = umls_request(descendants)
      descendants_list = process_hierarchial_term(descendants_obj["result"])

    concept_dict = {}
    if "result" in concepts_obj:
      concepts = concepts_obj["result"]["results"]

      for concept in concepts:
        concept_dict[concept["ui"]] = {"concept_uri": concept["uri"]}

      # Get defintions (include only English defintions) MRDEF
      for cui in concept_dict:
        concept_obj = umls_request(concept_dict[cui]["concept_uri"])

        if "result" in concept_obj:
          cui_concept_obj = concept_obj["result"]
          semantic_types = [s["name"] for s in cui_concept_obj["semanticTypes"]]
          if len(semantic_types) == 1:
            concept_dict[cui]["semantic_type"] = semantic_types[0]
          else:
            concept_dict[cui]["semantic_type"] = semantic_types

          concept_dict[cui]["definitions"] = {}
          if "definitions" in cui_concept_obj:

            if cui_concept_obj["definitions"] != "NONE":
              defintions_obj = umls_request(cui_concept_obj["definitions"])
              for result in defintions_obj["result"]:
                vocabulary = result["rootSource"]
                if languages[vocabulary] == "English":
                  concept_dict[cui]["definitions"][vocabulary] = result["value"]

  return {"code": code, "name": name, "vocabulary": source, "concepts": concept_dict,
          "attributes": attributes_dict, "relationships": relationship_list,
          "parents": parent_list, "children": children_list,
          "ancesotors": ancestors_list, "descendants": descendants_list}

In [9]:
vocabularies = get_vocabularies()
vocabularies

{'AIR': 'AI/RHEUM, 1993',
 'CST': 'COSTART, 1995',
 'DXP': 'DXplain, 1994',
 'LCH': 'Library of Congress Subject Headings, 1990',
 'MCM': 'McMaster University Epidemiology Terms, 1992',
 'SNM': 'SNOMED-2, 2',
 'SNMI': 'SNOMED International, 1998',
 'WHO': 'WHO Adverse Reaction Terminology, 1997',
 'ULT': 'UltraSTAR, 1993',
 'ICD10': 'ICD10, 1998',
 'ICPC': 'International Classification of Primary Care, 1993',
 'QMR': 'Quick Medical Reference (QMR), 1996',
 'RCD': 'Clinical Terms Version 3 (CTV3) (Read Codes), 1999',
 'PPAC': 'Pharmacy Practice Activity Classification, 1998',
 'AOD': 'Alcohol and Other Drug Thesaurus, 2000',
 'BI': 'Beth Israel Vocabulary, 1.0',
 'RCDAE': 'Read thesaurus, American English Equivalents, 1999',
 'RCDSA': 'Read thesaurus Americanized Synthesized Terms, 1999',
 'RCDSY': 'Read thesaurus, Synthesized Terms, 1999',
 'ICD10AE': 'ICD10, American English Equivalents, 1998',
 'DMDICD10': 'German translation of ICD10, 1995',
 'DMDUMD': 'German translation of UMDNS, 

In [10]:
%%time
get_code_source_information("J00", "ICD10CM")

CPU times: user 96.2 ms, sys: 10.7 ms, total: 107 ms
Wall time: 11.6 s


{'code': 'J00',
 'name': 'Acute nasopharyngitis [common cold]',
 'vocabulary': 'ICD10CM',
 'concepts': {'C0009443': {'concept_uri': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0009443',
   'semantic_type': 'Disease or Syndrome',
   'definitions': {'CSP': 'catarrhal disorder of the upper respiratory tract, which may be viral or a mixed infection; marked by acute coryza, slight rise in temperature, chilly sensations, and general indisposition.',
    'MEDLINEPLUS': '<h3>What is the common cold?</h3> <p>The common cold is a mild infection of your upper respiratory tract (which includes your nose and throat). Colds are probably the most common illness. Adults have an average of 2-3 colds per year, and children have even more. Colds are more common in the winter and spring, but you can get them at any time.</p> <h3>What causes the common cold?</h3> <p>More than 200 different viruses can cause a cold, but rhinoviruses are the most common type. The viruses that cause colds are very co

In [11]:
#list(genai.list_models())

## Helper CSV tables

In [12]:
import pandas as pd
ccsr_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/ccsr_codes.csv")
ccsr_df

,code,code_type,description
0,BLD001,CCSR_ICD10CM,Nutritional anemia
1,BLD002,CCSR_ICD10CM,Hemolytic anemia
2,BLD003,CCSR_ICD10CM,Aplastic anemia
3,BLD004,CCSR_ICD10CM,Acute posthemorrhagic anemia
4,BLD005,CCSR_ICD10CM,Sickle cell trait/anemia
...,...,...,...
549,SYM016,CCSR_ICD10CM,Other general signs and symptoms
550,SYM017,CCSR_ICD10CM,Abnormal findings without diagnosis
551,SYM018,CCSR_ICD10CM,Prediabetes
552,XXX000,CCSR_ICD10CM,Unacceptable PDX


In [13]:
full_icd10cm_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/umls_ohdsi_icd10_pt.csv")
full_icd10cm_df[full_icd10cm_df["TTY"] == "HT"]

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,...,SRL,SUPPRESS,CVF,dummy,STYs,PREFIX_CODE,ICD10CMCode,concept_code,concept_id,domain_id
5,C0002438,ENG,P,L0002438,PF,S0012920,N,A17850106,NaN,NaN,...,4,N,256.0,NaN,['Disease or Syndrome'],A06,A06,A06,1567245,Condition
11,C0002962,ENG,P,L0002962,VC,S0355458,N,A17826307,NaN,NaN,...,4,N,256.0,NaN,['Sign or Symptom'],I20,I20,I20,1569125,Condition
17,C0003705,ENG,S,L0823983,PF,S1047525,N,A17835348,NaN,NaN,...,4,N,256.0,NaN,['Injury or Poisoning'],T63,T633,T63.3,1575080,Condition
18,C0003742,ENG,P,L0003742,VC,S0356512,N,A17838687,NaN,NaN,...,4,N,256.0,NaN,['Disease or Syndrome'],H18,H1841,H18.41,45600876,Condition
19,C0003803,ENG,S,L0078983,VC,S0470971,N,A17815974,NaN,NaN,...,4,N,256.0,NaN,['Congenital Abnormality'],Q07,Q070,Q07.0,1572052,Condition
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97516,C5819435,ENG,P,L18773173,VC,S22498362,Y,A35632235,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],W44,W44E4,W44.E4,786274,Condition
97535,C5889707,ENG,S,L19377796,PF,S23151648,Y,A36508591,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],T45,T45A,T45.A,1102781,Observation
97556,C5926696,ENG,P,L19377687,VC,S23151191,Y,A36508788,NaN,NaN,...,4,N,256.0,NaN,['Acquired Abnormality'],K60,K6031,K60.31,1102697,Condition
97569,C5926771,ENG,P,L19377856,VC,S23151635,Y,A36508811,NaN,NaN,...,4,N,NaN,NaN,['Injury or Poisoning'],T45,T45AX3,T45.AX3,1102791,Condition


In [14]:
ht_range_terms_icd10cm_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/ht_range_terms_icd10cm.csv")
ht_range_terms_icd10cm_df

,CUI,CODE,STR
0,C0178238,A00-A09,Intestinal infectious diseases (A00-A09)
1,C0694449,A00-B99,Certain infectious and parasitic diseases (A00...
2,C0041296,A15-A19,Tuberculosis (A15-A19)
3,C0348110,A20-A28,Certain zoonotic bacterial diseases (A20-A28)
4,C2939130,A30-A49,Other bacterial diseases (A30-A49)
...,...,...,...
313,C0582114,Z66-Z66,Do not resuscitate status (Z66)
314,C2911644,Z67-Z67,Blood type (Z67)
315,C2240399,Z68-Z68,Body mass index [BMI] (Z68)
316,C0178343,Z69-Z76,Persons encountering health services in other ...


In [15]:
prefix_codes_icd10cm_expanded_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/prefix_codes_icd10cm_expanded.csv")
prefix_codes_icd10cm_expanded_df

,CUI,CODE,STR,PREFIX_CODE
0,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A00
1,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A01
2,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A02
3,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A03
4,C0178238,A00-A09,Intestinal infectious diseases (A00-A09),A04
...,...,...,...,...
5407,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z95
5408,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z96
5409,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z97
5410,C0478618,Z77-Z99,Persons with potential health hazards related ...,Z98


In [16]:
icd10cm_concept_map_snomed_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/icd10cm_concept_map_snomed.csv")
icd10cm_concept_map_snomed_df

,concept_id,concept_name,domain_id,concept_code,domain_id.1,concept_class_id,vocabulary_id,mapped_concept_id,mapped_concept_name,mapped_domain_id,mapped_concept_code,mapped_concept_class_id,mapped_vocabulary_id
0,35206181,Malignant neoplasm of overlapping sites of larynx,Condition,C32.8,Condition,4-char billing code,ICD10CM,22839,Overlapping malignant neoplasm of larynx,Condition,109369002,Disorder,SNOMED
1,35210612,Other neonatal hypoglycemia,Condition,P70.4,Condition,4-char billing code,ICD10CM,23034,Neonatal hypoglycemia,Condition,52767006,Disorder,SNOMED
2,35205603,Chlamydial infection of pharynx,Condition,A56.4,Condition,4-char billing code,ICD10CM,23137,Chlamydial pharyngitis,Condition,232403001,Disorder,SNOMED
3,1572082,Congenital malformations of esophagus,Condition,Q39,Condition,3-char nonbill code,ICD10CM,23868,Congenital anomaly of esophagus,Condition,69771008,Disorder,SNOMED
4,35210926,Other congenital malformations of esophagus,Condition,Q39.8,Condition,4-char billing code,ICD10CM,23868,Congenital anomaly of esophagus,Condition,69771008,Disorder,SNOMED
...,...,...,...,...,...,...,...,...,...,...,...,...,...
127981,45596736,"Pathological fracture in neoplastic disease, l...",Condition,M84.564G,Condition,7-char billing code,ICD10CM,45766973,Pathologic fracture of fibula at site of neoplasm,Condition,704252008,Disorder,SNOMED
127982,45601603,"Pathological fracture in neoplastic disease, r...",Condition,M84.551G,Condition,7-char billing code,ICD10CM,45766974,Pathologic fracture of femur at site of neoplasm,Condition,704253003,Disorder,SNOMED
127983,45606330,Age-related osteoporosis with current patholog...,Condition,M80.041K,Condition,7-char billing code,ICD10CM,45767040,Osteoporotic fracture of hand,Condition,704333004,Disorder,SNOMED
127984,45572557,Age-related osteoporosis with current patholog...,Condition,M80.022K,Condition,7-char billing code,ICD10CM,45767042,Osteoporotic fracture of humerus,Condition,704335006,Disorder,SNOMED


## GenAI Wrapper Functions

In [17]:
"""Code for interacting with GenAI prompts"""


def load_multiple_objects_into_model(objects_dict, prompt, model="models/gemini-2.5-flash",
                                    count_tokens=None,
                                    serialization="json", structured_responses=False):
  """
    Loads multiple object into a generative model. Objects are passed in a dictionary
    where the key is used in the prompt before the value of the object.
  """
  model_obj = genai.GenerativeModel(model)

  object_prompt = ""
  for key in objects_dict:
    object_prompt += f"{key} - Object serialzed in '{serialization}'\n"
    if serialization == "json":
      object_prompt += json.dumps(objects_dict[key]) + "\n\n"
    else:
      object_prompt += str(objects_dict[key]) + "\n\n"

  full_prompt = f"{object_prompt}\n\n{prompt}"

  if count_tokens:
    response = model_obj.count_tokens(full_prompt)
    print(f"Total prompt size: {len(full_prompt)} which translates into {response}")
    return response
  else:
    response = model_obj.generate_content(full_prompt)

    if structured_responses:
      try:
          return json.loads(response.text)
      except json.JSONDecodeError:

        return json.loads(response.text.split("```")[1][5:])
      except json.DecoderError:
        print(response.text.split("```")[1])
        raise RuntimeError("JSON Decoder Error")
    else:
      return response.text

def evaluate_code(code, source="ICD10CM", model="models/gemini-2.5-flash", prompt="Can you summarize how the code should be used?",
                  token_size_only=False, structured_responses=False):
  """Gets context for a UMLS sourced code and evaluates a prompt against it"""

  return load_multiple_objects_into_model(model=model, objects_dict={source: get_code_source_information(code, source)},
                                  prompt=prompt, structured_responses=structured_responses,
                                  count_tokens=token_size_only)




## GenAI Examples

### CCSR examples

#### CCSR Pipeline Example to OHDSI Standardized Concepts

In [18]:
import io
from re import split
import pprint

def pull_out_ccsr_mapping(umls_dict):
  streamlined_dict = {}
  for ccsr_code in umls_dict:
    streamlined_dict[ccsr_code] = []
    for relationship in umls_dict[ccsr_code]["relationships"]:
      if relationship['relationship_type_description'] == 'default_inpatient_classification_of':
        streamlined_dict[ccsr_code] += [relationship]
  return streamlined_dict


def ccsr_to_ICD10CM(overall_ccsr_criteria, filter_icd10cm_criteria, output_criteria='Return ICD10CM codes with text descriptions in a CSV file format and quote all text fields with a " quote.  The header for CSV file should be the columns of "code,description". Explain why certain codes were exlcuded from the output.'):
  ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}

  prompt_1 = overall_ccsr_criteria + " Only return CCSR code as elements in a JSON list."

  print(f"Prompt: {prompt_1}")
  print("")

  ccsr_codes = load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt=prompt_1, structured_responses=True)

  print(f"Found the following CCSR codes: {ccsr_codes}")
  print("Description of CCSR Codes:")
  pprint.pprint(
  {pair["code"]: pair["description"] for pair in ccsr_dict["CCSR"] if pair["code"] in ccsr_codes})

  print("")

  ccsr_umls_dict = {c: get_code_source_information(c, "CCSR_ICD10CM") for c in ccsr_codes}
  streamlined_ccsr_umls_dict = pull_out_ccsr_mapping(ccsr_umls_dict)

  print(f"Character length of JSON structure dumped: {len(json.dumps(streamlined_ccsr_umls_dict))}")


  print("")

  prompt_2 = filter_icd10cm_criteria + output_criteria

  print(f"Prompt: {prompt_2}")

  print("")

  response = load_multiple_objects_into_model(objects_dict=streamlined_ccsr_umls_dict, prompt=prompt_2, structured_responses=False)

  return response


def extract_csv_codes_to_df(output):
  """Extracts CSV files from returned results"""
  split_output = output.split("```")

  for section in split_output:
    if section[0:3] == "csv":
      text_output = section[3:].strip()
      df = pd.read_csv(io.StringIO(text_output))
      columns = df.columns
      df.columns = [c.lower() for c in columns]
      return df.drop_duplicates()
  return None


In [19]:
examples = {
    "diabetes with vision complications": {
        "ccsr_prompt": "Find CCSR codes related to complications in diabetes.",
        "filter_prompt": "Return ICD10CM codes that are associated with complications to vision.",
        "mapping_filter_prompt": "filter codes to vision side effects from diabetes"
    },
    "obesity": {
        "ccsr_prompt": "Find CCSR codes related to obesity.",
        "filter_prompt": "Return ICD10CM codes that are associated with obesity.",
        "mapping_filter_prompt": "filter codes to obesity and obesitiy side effects"
        },
    "heart failure": {
        "ccsr_prompt": "Find CCSR codes related to heart failure.",
        "filter_prompt": "Return ICD10CM codes that are associated with heart failure.",
        "mapping_filter_prompt": "filter codes to heart failure and heart failure side effects"
    },
    "ischemic stroke": {
        "ccsr_prompt": "Find CCSR codes related to stroke and acute effects resulting from the stroke.",
        "filter_prompt": "Return ICD10CM codes for ischemic stroke. Exclude codes for late effects.",
        "mapping_filter_prompt": "filter codes to ischemic stroke remove codes only including occlusions and thrombosis."

    }
}

In [20]:
prompt_example = "diabetes with vision complications" # @param ["diabetes with vision complications","obesity","heart failure", "ischemic stroke"] {"allow-input":true}

ccsr_prompt = examples[prompt_example]["ccsr_prompt"]
filter_prompt = examples[prompt_example]["filter_prompt"]
mapping_filter_prompt = examples[prompt_example]["mapping_filter_prompt"]
prompt_find_ccsr_codes = ccsr_prompt
prompt_filter_ICD10CM_codes = filter_prompt


In [21]:
ccsr_codes = ccsr_to_ICD10CM(prompt_find_ccsr_codes, prompt_filter_ICD10CM_codes)
print(ccsr_codes)

Prompt: Find CCSR codes related to complications in diabetes. Only return CCSR code as elements in a JSON list.

Found the following CCSR codes: ['END003', 'PRG019']
Description of CCSR Codes:
{'END003': 'Diabetes mellitus with complication',
 'PRG019': 'Diabetes or abnormal glucose tolerance complicating pregnancy; '
           'childbirth; or the puerperium'}

Character length of JSON structure dumped: 127245

Prompt: Return ICD10CM codes that are associated with complications to vision.Return ICD10CM codes with text descriptions in a CSV file format and quote all text fields with a " quote.  The header for CSV file should be the columns of "code,description". Explain why certain codes were exlcuded from the output.

```csv
"code","description"
"E08.349","Diabetes mellitus due to underlying condition with severe nonproliferative diabetic retinopathy without macular edema"
"E08.351","Diabetes mellitus due to underlying condition with proliferative diabetic retinopathy with macular ede

In [22]:
codes_df = extract_csv_codes_to_df(ccsr_codes)

In [23]:
merged_codes_df = codes_df.merge(full_icd10cm_df, left_on="code", right_on="concept_code")[["concept_id", "code", "description", "domain_id"]].drop_duplicates().sort_values("code")
merged_codes_df

,concept_id,code,description,domain_id
2,45600633,E08.321,Diabetes mellitus due to underlying condition ...,Condition
4,45600634,E08.329,Diabetes mellitus due to underlying condition ...,Condition
7,45557107,E08.331,Diabetes mellitus due to underlying condition ...,Condition
5,45566723,E08.339,Diabetes mellitus due to underlying condition ...,Condition
6,45581342,E08.341,Diabetes mellitus due to underlying condition ...,Condition
...,...,...,...,...
260,37200308,E13.37X1,Other specified diabetes mellitus with diabeti...,Condition
261,37200309,E13.37X2,Other specified diabetes mellitus with diabeti...,Condition
259,37200310,E13.37X3,Other specified diabetes mellitus with diabeti...,Condition
256,37200311,E13.37X9,Other specified diabetes mellitus with diabeti...,Condition


In [24]:
codes_df[~codes_df["code"].isin(merged_codes_df["code"])].sort_values("code")

,code,description


In [25]:
filtered_codes_snomed_df = icd10cm_concept_map_snomed_df.merge(merged_codes_df, on="concept_id")[["mapped_concept_id", "mapped_concept_code", "mapped_concept_name", "mapped_vocabulary_id", "mapped_domain_id"]].drop_duplicates()
filtered_codes_snomed_df

,mapped_concept_id,mapped_concept_code,mapped_concept_name,mapped_vocabulary_id,mapped_domain_id
0,195771,8801005,Secondary diabetes mellitus,SNOMED,Condition
7,378743,312903003,Mild nonproliferative retinopathy due to diabe...,SNOMED,Condition
19,4174977,4855003,Retinopathy due to diabetes mellitus,SNOMED,Condition
22,4225656,421920002,Cataract due to diabetes mellitus type 1,SNOMED,Condition
23,37016180,138891000119109,Moderate nonproliferative retinopathy due to t...,SNOMED,Condition
33,37016356,368711000119106,Mild nonproliferative retinopathy due to secon...,SNOMED,Condition
43,380096,59276001,Proliferative retinopathy due to diabetes mell...,SNOMED,Condition
97,380097,312912001,Macular edema due to diabetes mellitus,SNOMED,Condition
184,4227210,420789003,Retinopathy due to type 1 diabetes mellitus,SNOMED,Condition
186,43530685,1501000119109,Proliferative retinopathy due to type 2 diabet...,SNOMED,Condition


In [29]:
mapped_filter_prompt = "Filter the JSON input for mapped_concept_name for those " + mapping_filter_prompt + " "
mapped_filter_prompt += '''Return results in a CSV file format where text values are escaped by " quotes. Explain why certain codes were excluded. CSV output must have the following columns: mapped_concept_id, mapped_concept_code, mapped_concept_name'''
mapped_codes_response = load_multiple_objects_into_model(objects_dict= {"mapped codes to SNOMED": filtered_codes_snomed_df.to_dict("records")}, prompt=mapped_filter_prompt)

print(f"Prompt: {mapped_filter_prompt}")
print("")
print(mapped_codes_response)

Prompt: Filter the JSON input for mapped_concept_name for those filter codes to vision side effects from diabetes Return results in a CSV file format where text values are escaped by " quotes. Explain why certain codes were excluded. CSV output must have the following columns: mapped_concept_id, mapped_concept_code, mapped_concept_name

```csv
mapped_concept_id,mapped_concept_code,mapped_concept_name
378743,312903003,"Mild nonproliferative retinopathy due to diabetes mellitus"
4174977,4855003,"Retinopathy due to diabetes mellitus"
4225656,421920002,"Cataract due to diabetes mellitus type 1"
37016180,138891000119109,"Moderate nonproliferative retinopathy due to type 1 diabetes mellitus"
37016356,368711000119106,"Mild nonproliferative retinopathy due to secondary diabetes mellitus"
380096,59276001,"Proliferative retinopathy due to diabetes mellitus"
380097,312912001,"Macular edema due to diabetes mellitus"
4227210,420789003,"Retinopathy due to type 1 diabetes mellitus"
43530685,150100011

In [30]:
cleaned_mapped_snomed_codes_df = extract_csv_codes_to_df(mapped_codes_response)
cleaned_mapped_snomed_codes_df

,mapped_concept_id,mapped_concept_code,mapped_concept_name
0,378743,312903003,Mild nonproliferative retinopathy due to diabe...
1,4174977,4855003,Retinopathy due to diabetes mellitus
2,4225656,421920002,Cataract due to diabetes mellitus type 1
3,37016180,138891000119109,Moderate nonproliferative retinopathy due to t...
4,37016356,368711000119106,Mild nonproliferative retinopathy due to secon...
5,380096,59276001,Proliferative retinopathy due to diabetes mell...
6,380097,312912001,Macular edema due to diabetes mellitus
7,4227210,420789003,Retinopathy due to type 1 diabetes mellitus
8,43530685,1501000119109,Proliferative retinopathy due to type 2 diabet...
9,376114,312905005,Severe nonproliferative retinopathy due to dia...


In [31]:
# Output concept_ids so they can easily be pasted
[print(c) for c in cleaned_mapped_snomed_codes_df.mapped_concept_id.values.tolist()]
print("")
print(f"SNOMED codes generated for '{prompt_example}'. Paste the above codes into the ATLAS conceptset import tool.")

378743
4174977
4225656
37016180
37016356
380096
380097
4227210
43530685
376114
37016358
377552
43530656
45763584
376979
443733
4338901
45757435
45763583
45769873
45773064
443767
4290822
37016179
42538169
45770881
4221495
4266637
45770830

SNOMED codes generated for 'diabetes with vision complications'. Paste the above codes into the ATLAS conceptset import tool.


### ICD10CM

#### Header Term Pipeline

In [ ]:
def get_header_terms_dict(ht_df = ht_range_terms_icd10cm_df):
  header_terms_dict = ht_df.to_dict("records")

  return header_terms_dict


def get_icd10cm_codes_from_header_terms(header_terms, icd10cm_df=full_icd10cm_df):
  df_1 = icd10cm_df[["PREFIX_CODE", "CODE", "STR", "STYs"]].merge(prefix_codes_icd10cm_expanded_df, on="PREFIX_CODE")
  df_2 = df_1[df_1["CODE_y"].isin(header_terms)]

  df_3 = df_2[["CODE_x", "STR_x"]].drop_duplicates()

  df_3.columns = ["code", "description"]

  return df_3.to_dict("records")


def header_terms_to_icd10cm(prompt_find_header_terms, prompt_filter_codes):
  header_terms_dict = {"header_terms": get_header_terms_dict()}

  prompt_1 = prompt_find_header_terms + " Return header terms in a JSON list only."

  print(f"Prompt: {prompt_1}")
  print("")

  header_terms_obj = load_multiple_objects_into_model(objects_dict=header_terms_dict, prompt=prompt_1, structured_responses=True)

  header_terms_dict = {h["CODE"]: h["STR"] for h in header_terms_obj}

  header_terms = list(header_terms_dict)


  print(f"Found header terms:")
  pprint.pprint(header_terms_dict)
  print("")

  icd10cm_codes_list_dict = get_icd10cm_codes_from_header_terms(header_terms)

  prompt_2 = prompt_filter_codes + ' Return codes as CSV format with text values quoted in by " quotes. Place the CSV file in a ```csv ``` block. CSV should have a header row of code and description. Provide a summary why certain codes were excluded.'

  print(f"Prompt: {prompt_2}")
  print("")

  icd10cm_result_obj = load_multiple_objects_into_model({"ICD10CM": icd10cm_codes_list_dict}, prompt=prompt_2, structured_responses=False)

  return icd10cm_result_obj



In [ ]:
header_term_prompt = "Find header terms related to obesity and the effects of obesity and including BMI measure."
code_filter_prompt = "Only include ICD10CM related to obesity and adult bmi measures greater than 30 or in the 95th percentile"
mapped_filter_prompt = "filter codes to include those related to obesity"

result_header_term_obj = header_terms_to_icd10cm(header_term_prompt, code_filter_prompt)
print("")
print(result_header_term_obj)

In [ ]:
codes_df = extract_csv_codes_to_df(result_header_term_obj).drop_duplicates()
codes_df

In [ ]:
merged_codes_df = codes_df.merge(full_icd10cm_df, left_on="code", right_on="concept_code")[["concept_id", "code", "description", "domain_id"]].drop_duplicates().sort_values("code")
merged_codes_df

In [ ]:
filtered_codes_snomed_df = icd10cm_concept_map_snomed_df.merge(merged_codes_df, on="concept_id")[["mapped_concept_id", "mapped_concept_code", "mapped_concept_name", "mapped_vocabulary_id", "mapped_domain_id"]].drop_duplicates()
filtered_codes_snomed_df

In [ ]:
mapped_filter_prompt = "Filter the JSON input for mapped_concept_name for those " + mapped_filter_prompt + " "
mapped_filter_prompt += '''Return results in a CSV file format where text values are escaped by " quotes. Explain why certain codes were excluded.'''
mapped_codes_response = load_multiple_objects_into_model(objects_dict= {"mapped codes to SNOMED": filtered_codes_snomed_df.to_dict("records")}, prompt=mapped_filter_prompt)

print(f"Prompt: {mapped_filter_prompt}")
print("")
print(mapped_codes_response)

In [ ]:
cleaned_mapped_snomed_codes_df = extract_csv_codes_to_df(mapped_codes_response)
cleaned_mapped_snomed_codes_df

In [ ]:
# Output concept_ids so they can easily be pasted
[print(c) for c in cleaned_mapped_snomed_codes_df.mapped_concept_id.values.tolist()]
print("")
print(f"SNOMED codes'. Paste the above codes into the ATLAS conceptset import tool.")

4256640
433736
4097996
434005
4060985
4217557
4100857

SNOMED codes'. Paste the above codes into the ATLAS conceptset import tool.


#### Other ICD10CM Coding

In [ ]:
full_icd10cm_df[["concept_code", "STR", "concept_id"]][full_icd10cm_df["TTY"] == "PT"]

In [ ]:
icd10cm_pt_dict = full_icd10cm_df[["concept_code", "STR"]][full_icd10cm_df["TTY"] == "PT"].sort_values("concept_code").to_dict(orient="records")

In [ ]:
load_multiple_objects_into_model(objects_dict={"ICD10CM": icd10cm_pt_dict}, prompt="Find all ICD10CM codes for cancer originating in breast tissue. List results in CSV format and include the description and concept_id", count_tokens=True)

In [ ]:
t = full_icd10cm_df[["concept_code", "STR"]][full_icd10cm_df["TTY"] == "PT"].sort_values("concept_code").to_csv(index=False)

In [ ]:
load_multiple_objects_into_model(objects_dict={"ICD10CM (CSV format)": t}, count_tokens=True, prompt="")

### Elixhauser comorbidities

In [ ]:
elixhauser_codes_map_snomed_df = pd.read_csv("https://raw.githubusercontent.com/jhajagos/SupportingConceptSetGeneration/refs/heads/main/comorbidities/data/generated_from_nb/Build_ICD10CM_tables/elixhauser_codes_map_snomed.csv")
elixhauser_codes_map_snomed_df

In [ ]:
abridged_elixhauser_codes_map_snomed_df = elixhauser_codes_map_snomed_df[["code_class","concept_id","code", "concept_name", "mapped_concept_id", "mapped_vocabulary_id", "mapped_concept_code", "mapped_concept_name"]]
abridged_elixhauser_codes_map_snomed_df

In [ ]:
ab_al_dict = abridged_elixhauser_codes_map_snomed_df[abridged_elixhauser_codes_map_snomed_df["code_class"] == "alcohol"].to_dict(orient="records")
len(ab_al_dict)

In [ ]:
abridged_elixhauser_codes_map_snomed_df[abridged_elixhauser_codes_map_snomed_df["code_class"] == "alcohol"][["mapped_concept_id", "mapped_concept_code", "mapped_concept_name"]].drop_duplicates()

In [ ]:
print(load_multiple_objects_into_model(objects_dict={"ICD10CM mapped to SNOMED": ab_al_dict}, count_tokens=False, prompt="Generate a CSV list mapped codes with descriptions mapped concept_id that are not related to alcohol or effects of alcohol"))

In [ ]:
elixhauser_codes_map_snomed_df["code_class"].drop_duplicates()

In [ ]:
def reason_over_icd10_codes_mapped(code_class="wloss", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=""):
  mapped_dict = snomed_code_mapped_df[snomed_code_mapped_df["code_class"] == code_class].to_dict(orient="records")
  return load_multiple_objects_into_model(objects_dict={"ICD10CM mapped to SNOMED": mapped_dict}, count_tokens=False, prompt=prompt)

In [ ]:
prompt="Generate a CSV list mapped SNOMED codes with descriptions mapped concept_ids that are not related to weight loss"
print(reason_over_icd10_codes_mapped(code_class="wloss", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=prompt))

In [ ]:
prompt="""Generate a CSV list mapped SNOMED codes with descriptions mapped concept_ids
 and mapped_concept_codes that are not related to heart failure. Remove duplicates."""
print(reason_over_icd10_codes_mapped(code_class="chf", snomed_code_mapped_df=elixhauser_codes_map_snomed_df, prompt=prompt))

### Rxnorm

In [ ]:
print(evaluate_code("153165", "RXNORM"))

In [ ]:
print(evaluate_code("83367", "RXNORM", prompt="list all contrandications for this medication"))

### CPT

In [ ]:
print(evaluate_code("33361", "CPT"))

### LOINC

In [ ]:
print(evaluate_code("4548-4", "LNC"))

### HPO

In [ ]:
print(evaluate_code("HP:0033630", "HPO"))

This JSON object represents a **structured data record for the Human Phenotype Ontology (HPO) concept "Brain fog" (HP:0033630)**.

It's designed to provide a comprehensive, machine-readable overview of this specific phenotype, including its definitions, relationships to other concepts, and its position within the HPO hierarchy.

Here's how this "code" (data object) should be used:

1.  **Identify the HPO Term:**
    *   `"code": "HP:0033630"`: This is the unique identifier for "Brain fog" within the HPO.
    *   `"name": "Brain fog"`: This is the human-readable name of the phenotype.
    *   `"vocabulary": "HPO"`: Confirms that this record pertains to the Human Phenotype Ontology.

2.  **Access Definitions and Conceptual Mappings:**
    *   The `"concepts"` field links the HPO term to a broader medical concept using a UMLS Concept Unique Identifier (CUI).
        *   `"C0015676"`: This is the UMLS CUI for "Brain fog."
        *   `"concept_uri"`: Provides a direct link to the UMLs defi

In [ ]:
get_code_source_information("HP:0033630", "HPO")

{'code': 'HP:0033630',
 'name': 'Brain fog',
 'vocabulary': 'HPO',
 'concepts': {'C0015676': {'concept_uri': 'https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0015676',
   'semantic_type': 'Mental or Behavioral Dysfunction',
   'definitions': {'MSH': 'A condition of low alertness or cognitive impairment, usually associated with prolonged mental activities or stress.',
    'HPO': 'Brain fog is a type of transient cognitive dysfunction that comprises a constellation of symptoms that impair intellectual functioning to a level that interferes with daily activities, commonly including forgetfulness, mental slowness, difficulty thinking or focusing, a perceived slowing of mental processing speed, inability to find the right words, a sensation that the mind went blank or is cloudy". Brain fog tends to recur and may be triggered by factors such as physical fatigue, lack of sleep, and prolonged standing or may appear to occur spontaneously." [PMID:23999934, PMID:30452327, PMID:32857796, PMID

### OMIM

In [ ]:
print(evaluate_code("190070", "OMIM"))

### SNOMED

In [ ]:
print(evaluate_code("44054006", "SNOMEDCT_US"))

#### Other CCSR Examples

In [ ]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt="Can you summarize which CCSR code for diabetes or complications from diabetes?"))

Based on the provided CCSR codes and their descriptions, here are the codes related to diabetes or its complications:

**Directly Related to Diabetes (Diagnosis & Types):**

*   **END002**: Diabetes mellitus without complication
*   **END003**: Diabetes mellitus with complication
*   **END004**: Diabetes mellitus, Type 1
*   **END005**: Diabetes mellitus, Type 2
*   **END006**: Diabetes mellitus, due to underlying condition, drug or chemical induced, or other specified type
*   **PRG019**: Diabetes or abnormal glucose tolerance complicating pregnancy; childbirth; or the puerperium
*   **SYM018**: Prediabetes

**Common Complications of Diabetes (these conditions are often caused or exacerbated by diabetes, though they can have other causes too):**

*   **CIR026**: Peripheral and visceral vascular disease (common diabetic complication)
*   **CIR028**: Gangrene (severe complication often due to diabetic vascular disease)
*   **EYE005**: Retinal and vitreous conditions (would include diabe

In [ ]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
code_prompt="Can you summarize which CCSR code for diabetes or complications from diabetes? Only return the CCSR codes as JSON list."
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt=code_prompt, structured_responses=True))

['END002', 'END003', 'END004', 'END005', 'END006', 'PRG019', 'SYM018']


In [ ]:
ccsr_dict = {"CCSR": ccsr_df[["code","description"]].to_dict(orient="records")}
print(load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_dict, prompt="Can you summarize which CCSR for breast cancer?"))

Based on the provided CCSR codes, the following are related to breast cancer:

*   **NEO029**: "Breast cancer - ductal carcinoma in situ (DCIS)"
*   **NEO030**: "Breast cancer - all other types"


In [ ]:
prompt="Can you find which CCSR code are related to breast cancer. Return the CCSR codes only in a JSON list"
ccsr_codes = load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_dict, prompt=prompt, structured_responses=True)
ccsr_codes

['NEO029', 'NEO030']

In [ ]:
ccsr_umls_dict = {c: get_code_source_information(c, "CCSR_ICD10CM") for c in ccsr_codes}
print(load_multiple_objects_into_model(model="models/gemini-2.5-flash", objects_dict=ccsr_umls_dict,
                                      prompt="Return ICD10CM codes and descriptions in a CSV format where primary tumor occurred in the breast. Explain why certain codes were excluded"))

```csv
Code,Description
C50.119,Malignant neoplasm of central portion of unspecified female breast
C50.011,Malignant neoplasm of nipple and areola, right female breast
C50.021,Malignant neoplasm of nipple and areola, right male breast
C50.111,Malignant neoplasm of central portion of right female breast
C50.029,Malignant neoplasm of nipple and areola, unspecified male breast
C50.112,Malignant neoplasm of central portion of left female breast
C50.012,Malignant neoplasm of nipple and areola, left female breast
C50.019,Malignant neoplasm of nipple and areola, unspecified female breast
C50.022,Malignant neoplasm of nipple and areola, left male breast
C50.122,Malignant neoplasm of central portion of left male breast
C50.211,Malignant neoplasm of upper-inner quadrant of right female breast
C50.221,Malignant neoplasm of upper-inner quadrant of right male breast
C50.222,Malignant neoplasm of upper-inner quadrant of left male breast
C50.229,Malignant neoplasm of upper-inner quadrant of unspecifi

In [ ]:
import io
df = pd.read_csv(io.StringIO(evaluate_code("END004", "CCSR_ICD10CM", prompt='Return a CSV output (escape text fields with "") of ICD10CM codes and descriptions that are associated with complications to the eyes')))
df

,ICD10CM Code,Description
0,E10.331,Type 1 diabetes mellitus with moderate nonprol...
1,E10.341,Type 1 diabetes mellitus with severe nonprolif...
2,E10.311,Type 1 diabetes mellitus with unspecified diab...
3,E10.321,Type 1 diabetes mellitus with mild nonprolifer...
4,E10.339,Type 1 diabetes mellitus with moderate nonprol...
...,...,...
60,E10.37X1,Type 1 diabetes mellitus with diabetic macular...
61,E10.37X2,Type 1 diabetes mellitus with diabetic macular...
62,E10.3599,Type 1 diabetes mellitus with proliferative di...
63,E10.37X9,Type 1 diabetes mellitus with diabetic macular...


In [ ]:
# Output concept_ids so they can easily be pasted
[print(c) for c in cleaned_mapped_snomed_codes_df.mapped_concept_id.values.tolist()]
print("")
print(f"SNOMED codes generated for '{prompt_example}'. Paste the above codes into the ATLAS conceptset import tool.")

378743
4174977
4225656
37016180
37016356
380096
380097
4227210
43530685
376114
37016358
377552
43530656
45763584
376979
443733
4338901
45757435
45763583
45769873
45773064
443767
4290822
37016179
42538169
45770881
4221495
4266637
45770830

SNOMED codes generated for 'diabetes with vision complications'. Paste the above codes into the ATLAS conceptset import tool.


In [ ]:
print(load_multiple_objects_into_model(objects_dict=ccsr_dict, prompt="Can you summarize which CCSR code for diabetes or complications from diabetes?"))

The CCSR (Clinical Classifications Software Refined) codes that specifically refer to diabetes or its complications are primarily found under the "END" (Endocrine, Nutritional, and Metabolic Disorders) category, but also include other categories for common diabetic complications affecting different body systems.

Here's a summary:

**Primary Diabetes-Related Codes (END - Endocrine, Nutritional, and Metabolic Disorders):**

*   `END002`: Diabetes mellitus without complication
*   `END003`: Diabetes mellitus with complication
*   `END004`: Diabetes mellitus, Type 1
*   `END005`: Diabetes mellitus, Type 2
*   `END006`: Diabetes mellitus, due to underlying condition, drug or chemical induced, or other specified type

**Related Conditions:**

*   `SYM018`: Prediabetes (found under Symptoms, Signs, and Ill-defined Conditions)

**Diabetes Complicating Pregnancy:**

*   `PRG019`: Diabetes or abnormal glucose tolerance complicating pregnancy; childbirth; or the puerperium (found under Pregnancy

In [ ]:
print(evaluate_code("INJ031", "CCSR_ICD10CM"))

This JSON object defines a specific category within the **CCSR_ICD10CM (Clinical Classifications Software Refined for ICD-10-CM)** vocabulary.

Here's how this code (INJ031) should be used:

1.  **Identify and Classify "Allergic Reactions":**
    *   The `code`: **INJ031** is the unique identifier for this CCSR category.
    *   The `name`: **"Allergic reactions"** clearly states the clinical grouping represented by this code.
    *   **Purpose:** This code is used to aggregate and categorize various specific ICD-10-CM diagnoses into a broader, more manageable clinical concept of "Allergic reactions."

2.  **Understand the Clinical Definition:**
    *   The `concepts` section provides a standardized, authoritative definition of "Allergic reactions" from well-known medical terminologies (SNOMEDCT_US, NCI).
    *   **Purpose:** This ensures a consistent understanding of what constitutes an allergic reaction when using the INJ031 code, aiding in data interpretation and clinical consistenc

In [ ]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are related to the ear."))

H65.111,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), right ear"
H65.113,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), bilateral"
H65.114,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, right ear"
H65.119,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), unspecified ear"
H65.115,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, left ear"
H65.412,"Chronic allergic otitis media, left ear"
H65.413,"Chronic allergic otitis media, bilateral"
H65.116,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, bilateral"
H65.411,"Chronic allergic otitis media, right ear"
H65.117,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), recurrent, unspecified ear"
H65.112,"Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), left ear"
H65.419,"Chronic allergic otitis media, unspecifie

In [ ]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are related to the skin."))

```csv
ICD10CM Code,Description
L27.0,Generalized skin eruption due to drugs and medicaments taken internally
L23.81,Allergic contact dermatitis due to animal (cat) (dog) dander
L27.1,Localized skin eruption due to drugs and medicaments taken internally
L51.2,Toxic epidermal necrolysis [Lyell]
L51.9,Erythema multiforme, unspecified
L27.2,Dermatitis due to ingested food
L20.89,Other atopic dermatitis
L23.1,Allergic contact dermatitis due to adhesives
L23.2,Allergic contact dermatitis due to cosmetics
L23.4,Allergic contact dermatitis due to dyes
L23.3,Allergic contact dermatitis due to drugs in contact with skin
L23.7,Allergic contact dermatitis due to plants, except food
L23.6,Allergic contact dermatitis due to food in contact with the skin
L56.2,Photocontact dermatitis [berloque dermatitis]
L30.9,Dermatitis, unspecified
L20.9,Atopic dermatitis, unspecified
L20.81,Atopic neurodermatitis
L23.5,Allergic contact dermatitis due to other chemical products
L27.8,Dermatitis due to other subst

In [ ]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="How many ICD10CM codes are in this group?"))

There are 68 unique ICD-10-CM codes in this group.


In [ ]:
len(get_code_source_information("INJ031", "CCSR_ICD10CM")["relationships"])

138

In [ ]:
print(evaluate_code("INJ031", "CCSR_ICD10CM", prompt="Return a CSV list of the ICD10CM codes and descriptions and drop duplicate rows"))

```csv
ICD10CM_Code,Description
K90.41,Non-celiac gluten sensitivity
H01.112,Allergic dermatitis of right lower eyelid
H01.111,Allergic dermatitis of right upper eyelid
H01.113,Allergic dermatitis of right eye, unspecified eyelid
H01.114,Allergic dermatitis of left upper eyelid
H01.119,Allergic dermatitis of unspecified eye, unspecified eyelid
H01.131,Eczematous dermatitis of right upper eyelid
H01.132,Eczematous dermatitis of right lower eyelid
H01.135,Eczematous dermatitis of left lower eyelid
H01.136,Eczematous dermatitis of left eye, unspecified eyelid
H01.139,Eczematous dermatitis of unspecified eye, unspecified eyelid
H01.134,Eczematous dermatitis of left upper eyelid
H01.115,Allergic dermatitis of left lower eyelid
H01.116,Allergic dermatitis of left eye, unspecified eyelid
H01.133,Eczematous dermatitis of right eye, unspecified eyelid
H65.111,Acute and subacute allergic otitis media (mucoid) (sanguinous) (serous), right ear
H65.113,Acute and subacute allergic otitis media (muco

In [ ]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes that are associated with gestational diabetes"))

Based on the provided JSON object, which describes "Diabetes mellitus, Type 1" (code END004) and its related ICD-10-CM classifications, there are **no ICD-10-CM codes associated with gestational diabetes**.

The codes listed in the JSON relate to Type 1 diabetes and its complications, including pre-existing Type 1 diabetes in pregnancy. Gestational diabetes is a distinct diagnosis.


In [ ]:
print(evaluate_code("PREG", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes that are associated with gestational diabetes"))

Here is a CSV list of ICD-10-CM codes associated with **gestational diabetes**. These codes specify the type of control (diet/exercise, insulin, oral hypoglycemic, or unspecified) and the timing (pregnancy trimester, childbirth, or puerperium).

```csv
O24.411,O24.412,O24.413,O24.419,O24.42,O24.43,O24.441,O24.442,O24.443,O24.449,O24.45,O24.46,O24.471,O24.472,O24.473,O24.479,O24.48,O24.49,O24.491,O24.492,O24.493,O24.499
```

**Explanation of the Codes (for context, not part of the requested CSV):**

*   **O24.41-**: Gestational diabetes mellitus, controlled by diet and exercise
    *   O24.411: ...in pregnancy, first trimester
    *   O24.412: ...in pregnancy, second trimester
    *   O24.413: ...in pregnancy, third trimester
    *   O24.419: ...in pregnancy, unspecified trimester
    *   O24.42: ...in childbirth
    *   O24.43: ...in puerperium
*   **O24.44-**: Gestational diabetes mellitus, controlled by insulin
    *   O24.441: ...in pregnancy, first trimester
    *   O24.442: ...in 

In [ ]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a CSV list of ICD10CM codes and descriptions that are associated with pregnancy"))

```csv
ICD10CM Code,Description
O24.013,Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester
O24.012,Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester
O24.011,Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester
O24.02,Pre-existing type 1 diabetes mellitus, in childbirth
O24.03,Pre-existing type 1 diabetes mellitus, in the puerperium
O24.019,Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester
```


In [ ]:
print(evaluate_code("END004", "CCSR_ICD10CM", prompt="Return a JSON document of ICD10CM codes and descriptions that are associated with pregnancy and complications"))

```json
{
  "pregnancy_and_complications_codes": [
    {
      "code": "O24.013",
      "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester"
    },
    {
      "code": "O24.012",
      "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester"
    },
    {
      "code": "O24.011",
      "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester"
    },
    {
      "code": "O24.02",
      "description": "Pre-existing type 1 diabetes mellitus, in childbirth"
    },
    {
      "code": "O24.03",
      "description": "Pre-existing type 1 diabetes mellitus, in the puerperium"
    },
    {
      "code": "O24.019",
      "description": "Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester"
    }
  ]
}
```


In [ ]:
print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt="Return a CSV file of ICD10CM codes and descriptions related to pre-exisiting type 1 and/or type 2 diabetes. After generating the list explain why certain codes were excluded"))

```csv
ICD10CM Code,Description
"O24.011","Pre-existing type 1 diabetes mellitus, in pregnancy, first trimester"
"O24.012","Pre-existing type 1 diabetes mellitus, in pregnancy, second trimester"
"O24.013","Pre-existing type 1 diabetes mellitus, in pregnancy, third trimester"
"O24.019","Pre-existing type 1 diabetes mellitus, in pregnancy, unspecified trimester"
"O24.02","Pre-existing type 1 diabetes mellitus, in childbirth"
"O24.03","Pre-existing type 1 diabetes mellitus, in the puerperium"
"O24.111","Pre-existing type 2 diabetes mellitus, in pregnancy, first trimester"
"O24.112","Pre-existing type 2 diabetes mellitus, in pregnancy, second trimester"
"O24.113","Pre-existing type 2 diabetes mellitus, in pregnancy, third trimester"
"O24.119","Pre-existing type 2 diabetes mellitus, in pregnancy, unspecified trimester"
"O24.12","Pre-existing type 2 diabetes mellitus, in childbirth"
"O24.13","Pre-existing type 2 diabetes mellitus, in the puerperium"
```

### Explanation for Excluded Codes

T

In [ ]:
prompt="""Return a comma separated list of all the ICD10CM codes for type 1 or type 2 diabetes during pregnancy.
Codes in the list should be single quoted. Explain after listing the ICD10CM codes why certain codes
from the list were excluded."""

print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt=prompt))

The ICD-10-CM codes for type 1 or type 2 diabetes specifically during pregnancy are:

'O24.011', 'O24.012', 'O24.013', 'O24.019', 'O24.111', 'O24.112', 'O24.113', 'O24.119'

**Explanation for Excluded Codes:**

The following categories of codes were excluded based on the specific request for "type 1 or type 2 diabetes during pregnancy":

*   **Codes for Childbirth (O24.x2) or Puerperium (O24.x3):** The request specifically asked for codes "during pregnancy," referring to the antenatal period. Codes pertaining to childbirth (intrapartum) or the puerperium (postpartum period) were excluded as they represent distinct phases.
*   **Codes for Gestational Diabetes (O24.4xx):** The request explicitly specified "type 1 or type 2 diabetes." Gestational diabetes is a separate classification for diabetes that develops or is first recognized during pregnancy.
*   **Codes for Unspecified Diabetes (O24.3xx, O24.9xx):** These codes indicate that the specific type of diabetes (e.g., type 1 or type 2) 

In [ ]:
print(evaluate_code("PRG019", "CCSR_ICD10CM", prompt="When should I used this code?"))

The CCSR (Clinical Classifications Software Refined) code **PRG019 - Diabetes or abnormal glucose tolerance complicating pregnancy; childbirth; or the puerperium** should be used when the primary medical record contains an ICD-10-CM code that falls into this diagnostic category.

**In essence, you should use this CCSR code when a patient's medical documentation indicates any form of diabetes or abnormal glucose tolerance that is specifically impacting or being managed during:**

1.  **Pregnancy (any trimester):** This includes both pre-existing diabetes (Type 1, Type 2, or other) and gestational diabetes that develops during pregnancy.
2.  **Childbirth:** When diabetes or abnormal glucose tolerance is a factor during the labor and delivery process.
3.  **The Puerperium:** The postpartum period, usually up to 42 days after childbirth, where diabetes or abnormal glucose tolerance is still a relevant condition complicating the recovery.

**Specific ICD-10-CM codes that classify into PRG01